# Stage 8 — RAG Chain

**Project:** ResearchMate — Research Paper RAG Chatbot
**Goal of this notebook:** Build the actual chatbot — retriever + prompt + LLM wired together with LCEL — and test it with several questions.

**Before running:**
1. Upload `faiss_index.zip` (from Stage 6) to this Colab session.
2. Set up a free Groq API key as a Colab Secret named `GROQ_API_KEY` (see instructions in the chat message above this notebook).

## Cell 1 — Install packages

`langchain-groq` for the LLM. The rest are the same packages from Stages 6-7.

In [1]:
!pip install -q langchain-groq faiss-cpu langchain-community langchain-huggingface sentence-transformers langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## Cell 2 — Load the API key safely from Colab Secrets

This reads the key from Colab's Secrets manager (never typed into the notebook itself) and sets it as an environment variable, which is how `ChatGroq` expects to find it.

In [2]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
print("API key loaded from Colab Secrets.")

API key loaded from Colab Secrets.


## Cell 3 — Reload the vector store and create the retriever

Same steps as Stage 7: unzip the saved FAISS index, reload the embedding model, load the vector store, wrap it as a retriever (top-k = 3).

In [3]:
import zipfile
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

with zipfile.ZipFile("faiss_index.zip", "r") as zip_ref:
    zip_ref.extractall("faiss_index")

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = FAISS.load_local(
    "faiss_index",
    embedding_model,
    allow_dangerous_deserialization=True
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("Retriever ready. Total vectors:", vectorstore.index.ntotal)

/tmp/ipykernel_2406/2507533741.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Retriever ready. Total vectors: 20971


## Cell 4 — Create the LLM

`ChatGroq` with `llama-3.1-8b-instant` — fast, free-tier, and sufficient for grounded question-answering over short retrieved abstracts. `temperature=0` makes answers more consistent and less "creative," which is what we want for a research Q&A tool that should stick closely to the provided context.

In [9]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

print("LLM ready.")

LLM ready.


## Cell 5 — Build the prompt template

The system message sets the rules: answer only from the given context, don't invent facts, say so if the context is insufficient, and reference paper titles. `{context}` and `{question}` are placeholders that get filled in automatically when the chain runs.

In [10]:
from langchain_core.prompts import ChatPromptTemplate

SYSTEM_PROMPT = """You are ResearchMate, a research paper assistant.
Answer the user's question using ONLY the research paper context provided below.

Rules:
- Do not invent or assume any information that is not present in the context.
- If the context does not contain enough information to answer the question, say so clearly instead of guessing.
- When you use information from a paper, mention its title.
- Keep your answer clear and concise.

Context:
{context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}"),
])

print("Prompt template ready.")

Prompt template ready.


## Cell 6 — Helper function: format retrieved documents into one text block

The retriever returns a list of `Document` objects. The prompt just needs one plain text string, so this function joins the retrieved documents together, labeling each with its title so the LLM can reference them.

In [11]:
def format_docs(docs):
    formatted = []
    for doc in docs:
        formatted.append(f"Title: {doc.metadata['title']}\n{doc.page_content}")
    return "\n\n---\n\n".join(formatted)

# Quick test of the helper on its own
test_docs = retriever.invoke("black holes and gravitational waves")
print(format_docs(test_docs)[:500], "...")

Title: Los agujeros negros y las ondas del Doctor Einstein
Los agujeros negros y las ondas del Doctor Einstein

  We describe the main scientific developments that lead LIGO project to the
detection of the gravitational waves: general relativity, black holes and
gravitational waves predictions; numerical relativity and the collision and
coalescence simulations of binary black holes and the development of different
kind of gravitational wave detectors. Most important, this detection is
confirming ...


## Cell 7 — Build the RAG chain with LCEL

This is the core pipeline: `retriever | format_docs` handles the "get relevant papers, turn them into text" step, `RunnablePassthrough()` lets the original question pass through unchanged, the dict feeds both into the prompt, the prompt goes to the LLM, and `StrOutputParser()` extracts a plain string from the LLM's response.

In [12]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain ready.")

RAG chain ready.


## Cell 8 — Test the chatbot on several questions

We reuse the same questions from Stage 7, including the "transformer efficiency" case where retrieval returned a weak/off-target match — this tells us whether the LLM correctly recognizes and flags that the context doesn't really answer the question, as instructed in our prompt.

In [13]:
test_questions = [
    "How is Bayesian inference used in statistics?",
    "What research has been done on black holes and gravitational waves?",
    "What research has been done on transformer efficiency?",
]

for q in test_questions:
    print("QUESTION:", q)
    answer = rag_chain.invoke(q)
    print("\nANSWER:\n", answer)
    print("=" * 70)

QUESTION: How is Bayesian inference used in statistics?

ANSWER:
 Bayesian inference in statistics is used to update prior beliefs about unknown parameters with observed data, producing a posterior distribution that reflects both sources of information.  
- **Bayes’ theorem** is the core tool: it combines a prior distribution with the likelihood of the data to yield the posterior (see *Bayesian Methods in Cosmology*).  
- The posterior is then used for **parameter estimation** (e.g., Bayes estimates) and to construct **credible intervals** (again, *Bayesian Methods in Cosmology*).  
- For complex models, **numerical sampling** methods such as **Markov Chain Monte Carlo (MCMC)**—including Metropolis‑Hastings and Gibbs sampling—are employed to generate draws from the posterior (illustrated in *The Frechet distribution: Estimation and Application an Overview*).  
- Bayesian inference also supports **model comparison** through Bayesian model selection, contrasting it with classical p‑value

## What to check after running this notebook

- **Cell 2:** confirms your API key loaded without errors — if this fails, double check the secret name is exactly `GROQ_API_KEY` and notebook access is enabled.
- **Cell 7:** no errors building the chain.
- **Cell 8, question 1 & 2:** the answers should read naturally, reference real paper titles from our dataset, and actually address the question using the retrieved context.
- **Cell 8, question 3 (transformer efficiency):** this is the interesting one — given what Stage 7 showed us (weak/off-target retrieval for this question), does the LLM's answer **honestly flag** that the context doesn't really address "transformer efficiency" in the neural-network sense, rather than confidently making something up? This is the real test of our prompt's "say so if insufficient" instruction.

Paste back the 3 full answers from Cell 8, especially how the model handled the transformer efficiency question — then we'll move to Stage 9 (Improve Answer Format).